# Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV
from imblearn.over_sampling import SMOTE

In [36]:
df = pd.read_csv('heart_failure_dataset.csv')
df.head()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


# Data Preprocessing

In [37]:
df.isnull().sum()

age                         0
anaemia                     0
creatinine_phosphokinase    0
diabetes                    0
ejection_fraction           0
high_blood_pressure         0
platelets                   0
serum_creatinine            0
serum_sodium                0
sex                         0
smoking                     0
time                        0
DEATH_EVENT                 0
dtype: int64

In [38]:
x = df.iloc[:,:-1]
y = df.iloc[:,-1]
x

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8
...,...,...,...,...,...,...,...,...,...,...,...,...
294,62.0,0,61,1,38,1,155000.00,1.1,143,1,1,270
295,55.0,0,1820,0,38,0,270000.00,1.2,139,0,0,271
296,45.0,0,2060,1,60,0,742000.00,0.8,138,0,0,278
297,45.0,0,2413,0,38,0,140000.00,1.4,140,1,1,280


In [39]:
y

0      1
1      1
2      1
3      1
4      1
      ..
294    0
295    0
296    0
297    0
298    0
Name: DEATH_EVENT, Length: 299, dtype: int64

# Handling Imbalaced dataset

In [40]:
# To check if data is balaced or not
y.value_counts() 

DEATH_EVENT
0    203
1     96
Name: count, dtype: int64

# Split dataset into training and testing part

In [41]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state= 42)
print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)

(209, 12) (90, 12) (209,) (90,)


In [42]:
# Balancing the data
smote = SMOTE(random_state=42)
x_train_resampled, y_train_resampled = smote.fit_resample(x_train, y_train)

In [43]:
y_train_resampled.value_counts()

DEATH_EVENT
0    150
1    150
Name: count, dtype: int64

# Data Standardization

In [44]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

# ModelBuilding using RandomForestClassifier

In [45]:
model = RandomForestClassifier(class_weight='balanced', random_state=42)
model.fit(x_train_scaled, y_train) 
y_preds_training = model.predict(x_train_scaled)
accuracy_training = accuracy_score(y_train, y_preds_training)

y_preds_testing = model.predict(x_test_scaled)
accuracy_testing = accuracy_score(y_test, y_preds_testing)
print(accuracy_training)
print(accuracy_testing)

1.0
0.7666666666666667


# Hyperparameter Tunning
# improveing the accuracy of model using gridSearchCV

In [ ]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier())
])

# Define the hyperparameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'criterion': ['gini', 'entropy']
}

# Set up GridSearchCV
grid = GridSearchCV(estimator=model, param_grid=param_grid,
                    cv=5, scoring='accuracy', n_jobs=-1)

# Fit on your (balanced) data
grid.fit(x_train_resampled, y_train_resampled)

# Best parameters and score
print(grid.best_params_)
print(grid.best_score_)

In [ ]:
import pickle
# Get the best model
best_model = grid.best_estimator_

with open('model.pkl', 'wb') as f:
    pickle.dump({
        'model': best_model,
        'training_columns': list(x_train_resampled.columns)  # Save columns for validation
    }, f)

print("Model saved successfully using pickle!")

In [ ]:
import numpy as np
feature_names = ['age', 'anaemia', 'creatinine_phosphokinase', 'diabetes', 'ejection_fraction', 'high_blood_pressure', 'platelets','serum_creatinine', 'serum_sodium', 'sex', 'smoking', 'time']
#input_data = (82, 1, 379, 0, 50, 0, 47000, 1.3, 136, 1, 0, 13)

input_data = (49, 1, 80, 0, 30, 1, 427000, 1, 138, 0, 0, 12)
input_df = pd.DataFrame([input_data], columns=feature_names)
input_scaled = scaler.transform(input_df)
prediction = grid.best_estimator_.predict(input_scaled)
prediction[0]